> Generated from the matching `.md` file — **edit the markdown, not this notebook**, then run `python3 tools/build_notebooks.py`.
>
> Some examples are illustrative (Spark, external APIs, large models) and will not run without that service or package installed.

# Lesson 02 — Exploring a Dataset

**Goal:** know what you have, and what is wrong with it, in fifteen minutes.

## What you will learn

- The first six commands
- Reading distributions, not just means
- Finding the problems before they find you
- The exploration checklist

---

## A dataset to work with

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 5_000

orders = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "branch": rng.choice(["Zamalek", "Maadi", "Heliopolis", "Giza"],
                         n, p=[0.35, 0.3, 0.2, 0.15]),
    "product": rng.choice(["latte", "espresso", "tea", "cake", "juice"],
                          n, p=[0.3, 0.25, 0.2, 0.15, 0.1]),
    "quantity": rng.integers(1, 5, n),
    "unit_price": rng.choice([45.0, 35.0, 30.0, 60.0, 40.0], n),
    "ordered_at": pd.to_datetime("2026-01-01") + pd.to_timedelta(
        rng.integers(0, 180 * 24 * 60, n), unit="m"),
    "customer_id": rng.integers(1, 1_200, n),
})
orders["amount"] = orders["quantity"] * orders["unit_price"]

# a few realistic problems
orders.loc[rng.choice(n, 120, replace=False), "amount"] = np.nan
orders.loc[rng.choice(n, 15, replace=False), "amount"] = 50_000.0
orders = pd.concat([orders, orders.sample(40, random_state=1)], ignore_index=True)

orders.to_parquet("/tmp/orders.parquet")
print(orders.shape)

---

## The first six commands

In [ ]:
import pandas as pd

orders = pd.read_parquet("/tmp/orders.parquet")

print("1. shape:", orders.shape)
print("\n2. dtypes:")
print(orders.dtypes.to_string())
print("\n3. head:")
print(orders.head(3).to_string(index=False))
print("\n4. nulls per column:")
print(orders.isna().sum()[lambda s: s > 0].to_string())
print("\n5. duplicates:", int(orders.duplicated().sum()))
print("\n6. numeric summary:")
print(orders[["quantity", "unit_price", "amount"]].describe().round(1).to_string())

Four problems are already visible in that output, before any analysis:

1. **122 nulls in `amount`** — 2.4% of rows.
2. **40 exact duplicate rows.**
3. **The mean amount is 258.5 and the median is 105** — a mean 2.5× the median
   means a long tail or outliers.
4. **The maximum is 50,000** against a 75th percentile of 140. That is not a
   large order; it is a data error.

The `std` of 2,752 on a mean of 258 is the same signal: **when the standard
deviation is ten times the mean, look for outliers before anything else.**

---

## Distributions, not means

In [ ]:
import pandas as pd

orders = pd.read_parquet("/tmp/orders.parquet")
amount = orders["amount"].dropna()

print("mean:  ", round(amount.mean(), 1))
print("median:", round(amount.median(), 1))
print("\npercentiles:")
print(amount.quantile([0.01, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0]).round(1).to_string())

without_outliers = amount[amount < 1_000]
print(f"\nrows above 1,000: {len(amount) - len(without_outliers)}"
      f" ({(len(amount) - len(without_outliers)) / len(amount):.2%})")
print("mean without them:", round(without_outliers.mean(), 1))

Look at the 99th percentile and the maximum: **240 and 50,000**. Everything up
to 99% of the data is under 240, and then the top fifteen rows jump by a
factor of two hundred.

Those fifteen rows — **0.31% of the data** — moved the mean from 106.3 to
258.5. Every average, every "revenue per order", every forecast built on that
mean would have been 2.4 times reality.

**Always print percentiles, not just `mean()`.** The gap between the 99th
percentile and the maximum is where the errors hide.

---

## Categories and time

In [ ]:
import pandas as pd

orders = pd.read_parquet("/tmp/orders.parquet")

for column in ["branch", "product"]:
    counts = orders[column].value_counts()
    print(f"{column}: {orders[column].nunique()} distinct")
    print((counts / len(orders) * 100).round(1).to_string(), "\n")

print("date range:", orders["ordered_at"].min().date(), "to",
      orders["ordered_at"].max().date())
per_month = orders.set_index("ordered_at").resample("MS").size()
print("\nrows per month:")
print(per_month.to_string())

Two checks, both essential and both often skipped:

- **Category proportions.** A category that should be there and is not, or one
  taking 90% of the rows, tells you something about the data collection.
- **Rows per period.** A month with half the rows means a collection gap, not
  a business collapse — and confusing the two is how an analysis becomes
  wrong.

June has 790 rows and ends on the 29th, which is consistent with the others.
February's 771 is the lowest — and February is the shortest month, which is
the explanation rather than a finding. A month at 400 would have been the
story.

---

## The exploration checklist

In [ ]:
import pandas as pd

def explore(frame, name="dataset"):
    """The fifteen-minute profile, as a function."""
    report = {
        "rows": len(frame),
        "columns": frame.shape[1],
        "memory_mb": round(frame.memory_usage(deep=True).sum() / 1024**2, 1),
        "duplicate_rows": int(frame.duplicated().sum()),
        "columns_with_nulls": {c: int(v) for c, v in
                               frame.isna().sum().items() if v},
        "constant_columns": [c for c in frame.columns if frame[c].nunique() <= 1],
        "high_cardinality": {c: int(frame[c].nunique()) for c in frame.columns
                             if frame[c].nunique() > 0.5 * len(frame)},
    }
    numeric = frame.select_dtypes("number")
    report["possible_outliers"] = {
        c: int(((numeric[c] - numeric[c].median()).abs()
                > 10 * (numeric[c].quantile(0.75) - numeric[c].quantile(0.25))).sum())
        for c in numeric.columns
        if (numeric[c].quantile(0.75) - numeric[c].quantile(0.25)) > 0
    }
    return report

for key, value in explore(pd.read_parquet("/tmp/orders.parquet")).items():
    print(f"{key:<22}{value}")

Eight lines, and they name every problem in this dataset: 40 duplicates, 122
nulls, and 15 outliers in `amount`.

The `high_cardinality` entry flags `order_id` and `ordered_at`, which are
*supposed* to be nearly unique — the check is not saying they are wrong, it is
telling you which columns are identifiers rather than dimensions. A column you
expected to group by appearing there is the actual finding. Run this on **every**
dataset you receive, before you write a single aggregation.

And then do the thing no function can do: **read fifty rows with your eyes.**
Sort by the main measure, look at the top and the bottom, and see whether they
make sense.

---

## Common mistakes

| Mistake | What happens |
|---|---|
| `describe()` on the mean only | Outliers invisible |
| Not checking duplicates | Every total is inflated |
| Ignoring rows-per-period | A collection gap read as a business change |
| Trusting dtypes | A numeric column stored as text sorts "10" before "9" |
| Skipping the eyeball pass | Obvious nonsense survives to the report |
| Cleaning before exploring | You delete evidence of the real problem |

---

## Exercises

1. Run `explore()` on a dataset of yours and list every problem it finds.
2. Print percentiles for your main measure; how far is the max from the 99th?
3. Plot rows per month; are there collection gaps?
4. Sort by your main measure and read the top and bottom twenty rows.